# Bridge I Lab: Linear Algebra for Machine Learning

In this lab, we explore fundamental linear algebra concepts essential for machine learning: vectors as data points, norms and distance, matrix transformations, projections, and linear separability. Each section builds intuition through hands-on computation and visualization.

## Setup

This notebook uses numpy for numerical operations and matplotlib for visualization. All code is self-contained and reproducible with fixed random seeds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)

## Part 1: Vectors as Data Points

We represent each student by a 2D feature vector: (clicks per week, late submissions). We'll compute dot products between pairs of students to measure similarity.

In [ ]:
# Define three student profiles as 2D vectors
# (clicks per week, late submissions per semester)
alice = np.array([8.0, 2.0])      # Engaged, mostly on time
bob = np.array([3.0, 8.0])        # Disengaged, frequently late
charlie = np.array([7.0, 3.0])    # Mostly engaged, occasionally late

students = {'Alice': alice, 'Bob': bob, 'Charlie': charlie}

# Compute dot products (similarity measure)
print("Dot products (similarity):")
print(f"  Alice · Bob      = {np.dot(alice, bob):.1f}")
print(f"  Alice · Charlie  = {np.dot(alice, charlie):.1f}")
print(f"  Bob · Charlie    = {np.dot(bob, charlie):.1f}")
print()

# Interpret the results
print("Interpretation:")
print(f"  Alice and Charlie are similar (high dot product {np.dot(alice, charlie):.1f})")
print(f"  Alice and Bob are dissimilar (low dot product {np.dot(alice, bob):.1f})")
print(f"  Bob and Charlie are dissimilar (low dot product {np.dot(bob, charlie):.1f})")

In [ ]:
# Plot the vectors
fig, ax = plt.subplots(figsize=(8, 7))

# Plot vectors as arrows from origin
colors = {'Alice': 'steelblue', 'Bob': 'coral', 'Charlie': 'green'}
for name, vec in students.items():
    ax.arrow(0, 0, vec[0], vec[1], head_width=0.3, head_length=0.2,
             fc=colors[name], ec=colors[name], linewidth=2.5, label=name)

# Formatting
ax.set_xlim(-1, 10)
ax.set_ylim(-1, 9)
ax.set_xlabel('Clicks per Week', fontsize=12)
ax.set_ylabel('Late Submissions per Semester', fontsize=12)
ax.set_title('Student Profiles as 2D Feature Vectors', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper left')
ax.set_aspect('equal')

# Add angle annotation
theta_ac = np.arccos(np.dot(alice, charlie) / (np.linalg.norm(alice) * np.linalg.norm(charlie)))
ax.text(3, 1.5, f'Angle(Alice, Charlie)\n≈ {np.degrees(theta_ac):.1f}°', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Part 2: Norms and Distance

We compute L1 (Manhattan) and L2 (Euclidean) norms for each student vector, then calculate pairwise distances. We also visualize the unit circles for each norm.

In [ ]:
# Compute L1 and L2 norms
print("Norms:")
print()
for name, vec in students.items():
    l1_norm = np.linalg.norm(vec, ord=1)
    l2_norm = np.linalg.norm(vec, ord=2)
    print(f"{name}:")
    print(f"  L1 norm: {l1_norm:.2f}")
    print(f"  L2 norm: {l2_norm:.2f}")
    print()

In [ ]:
# Compute pairwise Euclidean distances
print("Pairwise Euclidean Distances:")
print(f"  d(Alice, Bob)      = {np.linalg.norm(alice - bob):.2f}")
print(f"  d(Alice, Charlie)  = {np.linalg.norm(alice - charlie):.2f}")
print(f"  d(Bob, Charlie)    = {np.linalg.norm(bob - charlie):.2f}")
print()
print("Interpretation:")
print(f"  Alice and Charlie are close (distance {np.linalg.norm(alice - charlie):.2f})")
print(f"  Alice and Bob are far apart (distance {np.linalg.norm(alice - bob):.2f})")

In [ ]:
# Plot unit circles for L1 and L2 norms
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

# L2 unit circle (Euclidean)
theta = np.linspace(0, 2*np.pi, 1000)
x_l2 = np.cos(theta)
y_l2 = np.sin(theta)
ax1.plot(x_l2, y_l2, linewidth=2.5, color='steelblue', label='L2 unit circle')
ax1.scatter([0], [0], c='black', s=50, zorder=5)
for name, vec in students.items():
    l2 = np.linalg.norm(vec, ord=2)
    scaled_vec = vec / l2
    ax1.arrow(0, 0, scaled_vec[0], scaled_vec[1], head_width=0.08, head_length=0.06,
             fc=colors[name], ec=colors[name], linewidth=2, alpha=0.7)
    ax1.text(scaled_vec[0]*1.2, scaled_vec[1]*1.2, name, fontsize=10, fontweight='bold')
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_xlabel('Feature 1 (normalized)', fontsize=11)
ax1.set_ylabel('Feature 2 (normalized)', fontsize=11)
ax1.set_title('L2 (Euclidean) Unit Circle', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')
ax1.legend(fontsize=10)

# L1 unit circle (Manhattan)
# Points satisfying |x| + |y| = 1
x_l1_1 = np.linspace(0, 1, 100)
x_l1_2 = np.linspace(1, 0, 100)
x_l1_3 = np.linspace(0, -1, 100)
x_l1_4 = np.linspace(-1, 0, 100)
y_l1_1 = 1 - x_l1_1
y_l1_2 = -(1 - x_l1_2)
y_l1_3 = -(1 + x_l1_3)
y_l1_4 = 1 + x_l1_4
ax2.plot(x_l1_1, y_l1_1, linewidth=2.5, color='coral', label='L1 unit circle')
ax2.plot(x_l1_2, y_l1_2, linewidth=2.5, color='coral')
ax2.plot(x_l1_3, y_l1_3, linewidth=2.5, color='coral')
ax2.plot(x_l1_4, y_l1_4, linewidth=2.5, color='coral')
ax2.scatter([0], [0], c='black', s=50, zorder=5)
for name, vec in students.items():
    l1 = np.linalg.norm(vec, ord=1)
    scaled_vec = vec / l1
    ax2.arrow(0, 0, scaled_vec[0], scaled_vec[1], head_width=0.08, head_length=0.06,
             fc=colors[name], ec=colors[name], linewidth=2, alpha=0.7)
    ax2.text(scaled_vec[0]*1.2, scaled_vec[1]*1.2, name, fontsize=10, fontweight='bold')
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_xlabel('Feature 1 (normalized)', fontsize=11)
ax2.set_ylabel('Feature 2 (normalized)', fontsize=11)
ax2.set_title('L1 (Manhattan) Unit Circle', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## Part 3: Matrix-Vector Multiplication as Transformation

We define a 2x2 matrix that stretches and rotates vectors. We apply it to points on the unit circle and visualize the transformation.

In [ ]:
# Define a 2x2 transformation matrix
# This matrix scales the first dimension by 2, rotates by 30°, and stretches the second dimension
theta_rot = np.radians(30)
scale_matrix = np.array([[1.5, 0], [0, 2.0]])
rot_matrix = np.array([[np.cos(theta_rot), -np.sin(theta_rot)],
                       [np.sin(theta_rot), np.cos(theta_rot)]])
A = rot_matrix @ scale_matrix

print("Transformation matrix A:")
print(A)
print()
print("This matrix applies scaling and rotation.")

In [ ]:
# Generate points on the unit circle
theta = np.linspace(0, 2*np.pi, 100)
unit_circle = np.array([np.cos(theta), np.sin(theta)])

# Apply the transformation: transformed = A @ unit_circle
transformed = A @ unit_circle

# Plot original and transformed
fig, ax = plt.subplots(figsize=(10, 8))

# Original unit circle
ax.plot(unit_circle[0], unit_circle[1], linewidth=2.5, color='steelblue', label='Original (unit circle)')
ax.scatter([1, 0], [0, 1], c='steelblue', s=60, zorder=5)  # Mark basis vectors

# Transformed circle
ax.plot(transformed[0], transformed[1], linewidth=2.5, color='coral', label='Transformed by A')

# Show where basis vectors map to
e1 = np.array([1, 0])
e2 = np.array([0, 1])
Ae1 = A @ e1
Ae2 = A @ e2

ax.arrow(0, 0, Ae1[0], Ae1[1], head_width=0.15, head_length=0.15,
        fc='darkred', ec='darkred', linewidth=2.5, alpha=0.7)
ax.arrow(0, 0, Ae2[0], Ae2[1], head_width=0.15, head_length=0.15,
        fc='darkgreen', ec='darkgreen', linewidth=2.5, alpha=0.7)
ax.text(Ae1[0]*1.15, Ae1[1]*1.15, 'A*e1', fontsize=11, fontweight='bold', color='darkred')
ax.text(Ae2[0]*1.15, Ae2[1]*1.15, 'A*e2', fontsize=11, fontweight='bold', color='darkgreen')

ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_title('Matrix-Vector Multiplication: Stretching and Rotating', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

print(f"e1 = {e1} maps to A*e1 = {Ae1}")
print(f"e2 = {e2} maps to A*e2 = {Ae2}")

## Part 4: Projection

We project a point onto a line defined by a direction vector. We visualize the original point, the projection, and the residual (orthogonal component).

In [ ]:
# Define a point and a direction vector (line through origin)
point = np.array([5.0, 3.0])
direction = np.array([2.0, 1.0])

# Normalize direction
direction_normalized = direction / np.linalg.norm(direction)

# Project point onto the line
# proj = (point · direction_normalized) * direction_normalized
scalar_proj = np.dot(point, direction_normalized)
projection = scalar_proj * direction_normalized

# Residual (orthogonal component)
residual = point - projection

print("Projection Analysis:")
print(f"Point: {point}")
print(f"Direction vector: {direction}")
print(f"Direction (normalized): {direction_normalized}")
print()
print(f"Scalar projection (length): {scalar_proj:.3f}")
print(f"Projection (point on line): {projection}")
print(f"Residual (orthogonal part): {residual}")
print()
print(f"Verification: projection · residual = {np.dot(projection, residual):.6f} (should be ~0)")

In [ ]:
# Visualize projection
fig, ax = plt.subplots(figsize=(10, 8))

# Draw the line (direction extended)
line_extent = 6
line_points = np.array([-line_extent * direction_normalized, line_extent * direction_normalized])
ax.plot(line_points[:, 0], line_points[:, 1], 'k--', linewidth=2, alpha=0.5, label='Line (direction)')

# Draw the original point
ax.scatter([point[0]], [point[1]], c='steelblue', s=150, zorder=5, label='Original point')
ax.text(point[0]+0.2, point[1]+0.3, 'Point', fontsize=11, fontweight='bold')

# Draw the projection
ax.scatter([projection[0]], [projection[1]], c='coral', s=150, zorder=5, label='Projection')
ax.text(projection[0]-0.5, projection[1]-0.5, 'Proj', fontsize=11, fontweight='bold', color='coral')

# Draw the residual (orthogonal part)
ax.arrow(projection[0], projection[1], residual[0], residual[1],
        head_width=0.2, head_length=0.15, fc='green', ec='green', linewidth=2.5, alpha=0.7)
ax.text(projection[0]+residual[0]/2+0.3, projection[1]+residual[1]/2+0.3, 'Residual', fontsize=10, color='green')

# Draw the projection line from point to projection
ax.plot([point[0], projection[0]], [point[1], projection[1]], 'g--', linewidth=2, alpha=0.5)

# Formatting
ax.set_xlim(-2, 6)
ax.set_ylim(-2, 5)
ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_title('Vector Projection onto a Line', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

## Part 5: Linear Separability

We generate two linearly separable clusters and find a separating hyperplane. Then we generate a non-separable (XOR) pattern and show why no line can separate it.

In [ ]:
# Generate two linearly separable clusters
np.random.seed(42)
n_samples = 50

# Class 0: cluster centered at (1, 1)
class_0 = np.random.randn(n_samples, 2) * 0.5 + np.array([1, 1])

# Class 1: cluster centered at (4, 4)
class_1 = np.random.randn(n_samples, 2) * 0.5 + np.array([4, 4])

# Find a separating hyperplane by hand
# Weight vector: direction perpendicular to the separating line
# A simple choice: normal vector between the two centers
center_0 = class_0.mean(axis=0)
center_1 = class_1.mean(axis=0)
w = center_1 - center_0  # Normal vector
w = w / np.linalg.norm(w)  # Normalize

# Bias: position the hyperplane between the two centers
midpoint = (center_0 + center_1) / 2
b = -np.dot(w, midpoint)

print(f"Separating hyperplane:")
print(f"  Weight vector w: {w}")
print(f"  Bias b: {b:.3f}")
print(f"  Decision function: f(x) = w·x + b")
print()
print(f"Classification:")
print(f"  If f(x) > 0: Class 1")
print(f"  If f(x) < 0: Class 0")

In [ ]:
# Plot linearly separable data with decision boundary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# LEFT: Linearly separable
ax1.scatter(class_0[:, 0], class_0[:, 1], c='red', s=50, alpha=0.6, label='Class 0', edgecolors='darkred')
ax1.scatter(class_1[:, 0], class_1[:, 1], c='blue', s=50, alpha=0.6, label='Class 1', edgecolors='darkblue')

# Draw the decision boundary: w·x + b = 0
x_min, x_max = class_0[:, 0].min() - 0.5, class_1[:, 0].max() + 0.5
y_min, y_max = class_0[:, 1].min() - 0.5, class_1[:, 1].max() + 0.5

# Compute decision boundary points
if abs(w[1]) > 0.01:
    x_boundary = np.linspace(x_min, x_max, 100)
    y_boundary = -(w[0] * x_boundary + b) / w[1]
    ax1.plot(x_boundary, y_boundary, 'k-', linewidth=2.5, label='Decision boundary')

ax1.set_xlim(x_min, x_max)
ax1.set_ylim(y_min, y_max)
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.set_title('Linearly Separable Data', fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)

# RIGHT: Non-separable (XOR) data
# Generate XOR pattern: (0,0)->0, (0,1)->1, (1,0)->1, (1,1)->0
xor_0 = np.array([[0.1 + 0.15*np.random.randn(), 0.1 + 0.15*np.random.randn()] for _ in range(25)])
xor_0 = np.vstack([xor_0, [[0.9 + 0.15*np.random.randn(), 0.9 + 0.15*np.random.randn()] for _ in range(25)]])

xor_1 = np.array([[0.1 + 0.15*np.random.randn(), 0.9 + 0.15*np.random.randn()] for _ in range(25)])
xor_1 = np.vstack([xor_1, [[0.9 + 0.15*np.random.randn(), 0.1 + 0.15*np.random.randn()] for _ in range(25)]])

ax2.scatter(xor_0[:, 0], xor_0[:, 1], c='red', s=50, alpha=0.6, label='Class 0', edgecolors='darkred')
ax2.scatter(xor_1[:, 0], xor_1[:, 1], c='blue', s=50, alpha=0.6, label='Class 1', edgecolors='darkblue')

# Try to draw a linear boundary (it will fail)
x_boundary_xor = np.linspace(-0.2, 1.2, 100)
# Arbitrary linear boundary (won't work)
y_boundary_xor = 0.5 * x_boundary_xor + 0.3
ax2.plot(x_boundary_xor, y_boundary_xor, 'k--', linewidth=2, alpha=0.5, label='No line works!')

ax2.set_xlim(-0.2, 1.2)
ax2.set_ylim(-0.2, 1.2)
ax2.set_xlabel('Feature 1', fontsize=12)
ax2.set_ylabel('Feature 2', fontsize=12)
ax2.set_title('Non-Separable (XOR) Data', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Observation:")
print("  XOR pattern cannot be separated by a single line (hyperplane).")
print("  This is why nonlinear classifiers (neural networks, kernel methods) are needed.")

## What to Notice

**Vectors as Data:**
- Dot products measure similarity; similar vectors have larger dot products
- The angle between vectors encodes their relationship

**Norms and Distance:**
- L2 (Euclidean) norm forms a circle; L1 (Manhattan) norm forms a diamond
- Different norms measure distance differently, but all capture magnitude
- Euclidean distance is intuitive for most ML applications

**Matrices as Transformations:**
- Matrices can stretch, rotate, shear, or reflect vectors
- The columns of a matrix show where the basis vectors map to
- The determinant measures how much a matrix scales areas (or volumes in higher dimensions)

**Projections:**
- Projecting a vector onto a line decomposes it into parallel and orthogonal components
- The residual is always perpendicular to the projection
- Least squares fitting solves projection problems

**Linear Separability:**
- Two clusters are linearly separable if a single hyperplane can partition them
- XOR is the canonical example of a non-linearly-separable problem
- Linear separability determines which classifiers (linear vs. nonlinear) can solve a problem